In [ ]:
!pip install -q transformers
!pip install -q datasets
!pip install -q sentencepiece
!pip install -q accelerate
!pip install -q evaluate
!pip install -q bert-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.5 MB/s eta 0:00:00


In [ ]:
import os
import pandas as pd
import numpy as np

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
DATA = "/content/drive/MyDrive/FIFI_Research/data"

train_df = pd.read_csv(
    os.path.join(DATA,"train.tsv"),
    sep="\t"
)

val_df = pd.read_csv(
    os.path.join(DATA,"val.tsv"),
    sep="\t"
)

print(train_df.shape)
print(val_df.shape)

(90000, 4)
(18000, 4)


In [ ]:
def build_prompt(row):

    return (
        "Recover the original scientific paper title.\n\n"
        f"Style: {row['category']}\n"
        f"Rewritten Title: {row['generated_title']}"
    )

train_df["input_text"]=train_df.apply(
    build_prompt,
    axis=1
)

val_df["input_text"]=val_df.apply(
    build_prompt,
    axis=1
)

train_df["target_text"]=train_df["original_title"]

val_df["target_text"]=val_df["original_title"]

train_df[
    [
        "input_text",
        "target_text"
    ]
].head()

,input_text,target_text
0,Recover the original scientific paper title.\n...,Improve High Level Classification with a More ...
1,Recover the original scientific paper title.\n...,Self-supervised Training of Proposal-based Seg...
2,Recover the original scientific paper title.\n...,Towards Spiral Brick Column Building Robots
3,Recover the original scientific paper title.\n...,Store Location Selection via Mining Search Que...
4,Recover the original scientific paper title.\n...,Potentials and Limitations of Deep Neural Netw...


In [ ]:
train_dataset=Dataset.from_pandas(

    train_df[
        [
            "input_text",
            "target_text"
        ]
    ]

)

val_dataset=Dataset.from_pandas(

    val_df[
        [
            "input_text",
            "target_text"
        ]
    ]

)

In [ ]:
MODEL_NAME = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME
)

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
MAX_INPUT=128
MAX_OUTPUT=64

def preprocess(examples):

    model_inputs=tokenizer(

        examples["input_text"],

        max_length=MAX_INPUT,

        truncation=True

    )

    labels=tokenizer(

        examples["target_text"],

        max_length=MAX_OUTPUT,

        truncation=True

    )

    model_inputs["labels"]=labels["input_ids"]

    return model_inputs

train_dataset=train_dataset.map(
    preprocess,
    batched=True
)

val_dataset=val_dataset.map(
    preprocess,
    batched=True
)

Map:   0%|          | 0/90000 [00:00<?, ? examples/s]

Map:   0%|          | 0/18000 [00:00<?, ? examples/s]

In [ ]:
data_collator=DataCollatorForSeq2Seq(

    tokenizer=tokenizer,

    model=model

)

In [ ]:
training_args = Seq2SeqTrainingArguments(

    output_dir="/content/flan_task2",

    per_device_train_batch_size=8,

    per_device_eval_batch_size=8,

    learning_rate=2e-5,

    num_train_epochs=1,

    logging_steps=200,

    save_strategy="epoch",

    predict_with_generate=True,

    fp16=True

)

In [ ]:
import transformers

print(transformers.__version__)

5.13.1


In [ ]:
trainer=Seq2SeqTrainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=val_dataset,

    data_collator=data_collator

)

In [ ]:
trainer.train()

Step,Training Loss
200,0.000000
400,0.000000
600,0.000000
800,0.000000
1000,0.000000
1200,0.000000
1400,0.000000
1600,0.000000
1800,0.000000
2000,0.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=11250, training_loss=0.0, metrics={'train_runtime': 1963.2358, 'train_samples_per_second': 45.843, 'train_steps_per_second': 5.73, 'total_flos': 5908246926925824.0, 'train_loss': 0.0, 'epoch': 1.0})

In [ ]:
MODEL_SAVE="/content/drive/MyDrive/FIFI_Research/models/flan_task2"

model.save_pretrained(MODEL_SAVE)

tokenizer.save_pretrained(MODEL_SAVE)

print("Model Saved Successfully")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model Saved Successfully


In [ ]:
!pip install -q transformers sentencepiece

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import pandas as pd

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM
)

In [ ]:
DATA="/content/drive/MyDrive/FIFI_Research/data"

val_df=pd.read_csv(
    os.path.join(DATA,"val.tsv"),
    sep="\t"
)

In [ ]:
def build_prompt(row):

    return (
        "Recover the original scientific paper title.\n\n"
        f"Style: {row['category']}\n"
        f"Rewritten Title: {row['generated_title']}"
    )

val_df["input_text"] = val_df.apply(
    build_prompt,
    axis=1
)

In [ ]:
MODEL_SAVE="/content/drive/MyDrive/FIFI_Research/models/flan_task2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_SAVE)

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_SAVE)

model.eval()

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


T5ForConditionalGeneration(
  (shared): Embedding(32128, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=768, out_features=2048, bias=False)
              (wi_1): Linear(in_features=768, out_features=2048, bias=False)
              (wo):

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)

print(device)

cuda


In [ ]:
from tqdm import tqdm

batch_size = 32

predictions = []

for i in tqdm(range(0, len(val_df), batch_size)):

    batch = val_df["input_text"].iloc[i:i+batch_size].tolist()

    inputs = tokenizer(
        batch,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=40
    )

    preds = tokenizer.batch_decode(
        outputs,
        skip_special_tokens=True
    )

    predictions.extend(preds)

val_df["prediction"] = predictions

val_df[
    [
        "generated_title",
        "prediction",
        "original_title"
    ]
].head(20)

100%|██████████| 563/563 [08:53<00:00,  1.05it/s]


,generated_title,prediction,original_title
0,Fully Convolutional Joint Detection and Regres...,Reproductive Limb-Pose Estimation from Depth I...,Preterm infants' limb-pose estimation from dep...
1,When Machines Argue: Teaching AI to Spot Fake ...,When Machines Argue: Teaching AI to Spot Fake ...,DSGAN: Generative Adversarial Training for Dis...
2,Optimized 2D Manifold Folding and Attribute Ma...,Optimized 2D Manifold Folding and Attribute Ma...,Folding-based compression of point cloud attri...
3,From Snapshot to Sawdust: Rebuilding Wooden Ob...,Recover the original scientific paper title. S...,Fabrication-Aware Reverse Engineering for Carp...
4,"A Survey of How Deep Learning Improves Image, ...","A Survey of How Deep Learning Improves Image, ...",Super-Resolution via Deep Learning
5,A Toolkit for Building Self-Navigating Robots:...,A Toolkit for Building Self-Navigating Robots:...,Autonomous Exploration Development Environment...
6,How to Share Machine Learning Resources Fairly...,Recover the original scientific paper title. S...,Ease.ml: Towards Multi-tenant Resource Sharing...
7,Clearing Up Shaky Video: How to Remove Atmosph...,Recover the original scientific paper title. S...,Atmospheric turbulence mitigation for sequence...
8,Educational Data Mining with Logistic Regressi...,Learning with Logistic Regression: Interpretab...,Modeling the EdNet Dataset with Logistic Regre...
9,How Do AI Language Models Learn Words? Compari...,A Study of the Language Learning of Machines,Word Acquisition in Neural Language Models


In [ ]:
submission = val_df[
    [
        "id",
        "category",
        "generated_title"
    ]
].copy()

submission["original_title"] = predictions

OUTPUT = "/content/drive/MyDrive/FIFI_Research/submissions"

os.makedirs(OUTPUT, exist_ok=True)

submission.to_csv(
    os.path.join(
        OUTPUT,
        "FutureMinds_task2_run1.tsv"
    ),
    sep="\t",
    index=False
)

print(submission.head())

   id    category                                    generated_title  \
0   0   technical  Fully Convolutional Joint Detection and Regres...   
1   1      catchy  When Machines Argue: Teaching AI to Spot Fake ...   
2   2   technical  Optimized 2D Manifold Folding and Attribute Ma...   
3   3      catchy  From Snapshot to Sawdust: Rebuilding Wooden Ob...   
4   4  accessible  A Survey of How Deep Learning Improves Image, ...   

                                      original_title  
0  Reproductive Limb-Pose Estimation from Depth I...  
1  When Machines Argue: Teaching AI to Spot Fake ...  
2  Optimized 2D Manifold Folding and Attribute Ma...  
3  Recover the original scientific paper title. S...  
4  A Survey of How Deep Learning Improves Image, ...  


In [ ]:
submission=val_df[
    [
        "id",
        "category",
        "generated_title"
    ]
].copy()

submission["original_title"]=predictions

OUTPUT="/content/drive/MyDrive/FIFI_Research/submissions"

os.makedirs(
    OUTPUT,
    exist_ok=True
)

submission.to_csv(

    os.path.join(

        OUTPUT,

        "FutureMinds_task2_run1.tsv"

    ),

    sep="\t",

    index=False

)

print(submission.head())

print(submission.shape)

   id    category                                    generated_title  \
0   0   technical  Fully Convolutional Joint Detection and Regres...   
1   1      catchy  When Machines Argue: Teaching AI to Spot Fake ...   
2   2   technical  Optimized 2D Manifold Folding and Attribute Ma...   
3   3      catchy  From Snapshot to Sawdust: Rebuilding Wooden Ob...   
4   4  accessible  A Survey of How Deep Learning Improves Image, ...   

                                      original_title  
0  Reproductive Limb-Pose Estimation from Depth I...  
1  When Machines Argue: Teaching AI to Spot Fake ...  
2  Optimized 2D Manifold Folding and Attribute Ma...  
3  Recover the original scientific paper title. S...  
4  A Survey of How Deep Learning Improves Image, ...  
(18000, 4)


In [ ]:
from bert_score import score

In [ ]:
!pip install -q evaluate

In [ ]:
import evaluate

bertscore = evaluate.load("bertscore")

In [ ]:
from collections import Counter
import numpy as np

def token_f1(pred, gt):

    pred_tokens = pred.lower().split()
    gt_tokens = gt.lower().split()

    common = Counter(pred_tokens) & Counter(gt_tokens)

    num_same = sum(common.values())

    if num_same == 0:
        return 0

    precision = num_same / len(pred_tokens)
    recall = num_same / len(gt_tokens)

    return 2 * precision * recall / (precision + recall)

f1_scores = [
    token_f1(p, g)
    for p, g in zip(val_df["prediction"], val_df["original_title"])
]

print("Mean Token F1:", np.mean(f1_scores))

Mean Token F1: 0.24853218631295132


In [ ]:
from collections import Counter

def token_f1(pred, gt):

    pred_tokens = pred.lower().split()
    gt_tokens = gt.lower().split()

    common = Counter(pred_tokens) & Counter(gt_tokens)

    num_same = sum(common.values())

    if num_same == 0:
        return 0

    precision = num_same / len(pred_tokens)
    recall = num_same / len(gt_tokens)

    return 2 * precision * recall / (precision + recall)

In [ ]:
for style in ["technical","accessible","catchy"]:

    subset = val_df[val_df["category"] == style]

    scores = []

    for pred, gt in zip(
        subset["prediction"],
        subset["original_title"]
    ):

        scores.append(
            token_f1(pred, gt)
        )

    print(style, np.mean(scores))

technical 0.39821152776694846
accessible 0.15549508860389127
catchy 0.1918899425680142


In [ ]:
val_df[
[
"generated_title",
"prediction",
"original_title"
]
].head(10)

,generated_title,prediction,original_title
0,Fully Convolutional Joint Detection and Regres...,Reproductive Limb-Pose Estimation from Depth I...,Preterm infants' limb-pose estimation from dep...
1,When Machines Argue: Teaching AI to Spot Fake ...,When Machines Argue: Teaching AI to Spot Fake ...,DSGAN: Generative Adversarial Training for Dis...
2,Optimized 2D Manifold Folding and Attribute Ma...,Optimized 2D Manifold Folding and Attribute Ma...,Folding-based compression of point cloud attri...
3,From Snapshot to Sawdust: Rebuilding Wooden Ob...,Recover the original scientific paper title. S...,Fabrication-Aware Reverse Engineering for Carp...
4,"A Survey of How Deep Learning Improves Image, ...","A Survey of How Deep Learning Improves Image, ...",Super-Resolution via Deep Learning
5,A Toolkit for Building Self-Navigating Robots:...,A Toolkit for Building Self-Navigating Robots:...,Autonomous Exploration Development Environment...
6,How to Share Machine Learning Resources Fairly...,Recover the original scientific paper title. S...,Ease.ml: Towards Multi-tenant Resource Sharing...
7,Clearing Up Shaky Video: How to Remove Atmosph...,Recover the original scientific paper title. S...,Atmospheric turbulence mitigation for sequence...
8,Educational Data Mining with Logistic Regressi...,Learning with Logistic Regression: Interpretab...,Modeling the EdNet Dataset with Logistic Regre...
9,How Do AI Language Models Learn Words? Compari...,A Study of the Language Learning of Machines,Word Acquisition in Neural Language Models


In [ ]:
val_df["prediction"].str.split().str.len().describe()

,prediction
count,18000.000000
mean,11.817056
std,3.521166
min,0.000000
25%,10.000000
50%,12.000000
75%,14.000000
max,30.000000


In [ ]:
val_df["prediction"].value_counts().head(20)

,count
prediction,
Recover the original scientific paper title. Style: accessible,815
Recover the original scientific paper title,318
Recover the original scientific paper title.,241
,96
Recover the original scientific paper title. Style: catchy,51
Recover the original scientific paper title. Style: technical,5
Recover the original scientific paper title. Style: accessible Recovered the original scientific paper title. Style: accessible,5
The New York Times,5
A New Approach to Deep Learning,4
